Building the Master datset for the model

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging

# Use repository-relative data folder instead of Google Drive
BASE_DIR = Path('..') / 'data'

# from google.colab import drive
# drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def prep_holidays(holiday_path: str) -> pd.DataFrame:
    df = pd.read_csv(holiday_path)
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    return df.groupby(['Year', 'Month']).size().reset_index(name='holiday_count')

def prep_seasonality(seasonality_path: str) -> pd.DataFrame:
    df = pd.read_csv(seasonality_path)
    mapping = {'Un-Favorable': 0, 'Moderate': 1, 'Favorable': 2}
    df['Seasonality_Score'] = df['Seasonality_Index'].map(mapping)
    return df

# ==========================================
# DIRECT EXECUTION
# ==========================================

print("Loading Data Layers...")

# Load Base & Temporal Data 
master = pd.read_csv(BASE_DIR / 'silver' / 'outlet_master.csv')
transactions = pd.read_csv(BASE_DIR / 'silver' / 'transactions_monthly_aggregated.csv')
spatial = pd.read_csv(BASE_DIR / 'gold' / 'gold_outlet_spatial_features.csv')
seasonality = prep_seasonality(str(BASE_DIR / 'silver' / 'distributor_seasonality.csv'))
monthly_holidays = pd.read_csv(BASE_DIR / 'silver' / 'monthly_holiday_count.csv')

print("Merging Feature Matrix...")

# Dynamic keys to handle casing variations
left_key = 'Outlet_ID' if 'Outlet_ID' in transactions.columns else 'outlet_id'
right_key = 'Outlet_ID' if 'Outlet_ID' in master.columns else 'outlet_id'

# Merge 1: Transactions & Master
df = pd.merge(transactions, master, left_on=left_key, right_on=right_key, how='inner')
if left_key != right_key and right_key in df.columns:
    df = df.drop(columns=[right_key])

# Merge 2: Spatial
spatial_key = 'Outlet_ID' if 'Outlet_ID' in spatial.columns else 'outlet_id'
df = pd.merge(df, spatial, left_on=left_key, right_on=spatial_key, how='left')
if left_key != spatial_key and spatial_key in df.columns:
    df = df.drop(columns=[spatial_key])

# Merge 3: Holidays
df = pd.merge(df, monthly_holidays, on=['Year', 'Month'], how='left')
df['holiday_count'] = df['holiday_count'].fillna(0)

# Merge 4: Seasonality
if 'Distributor_ID' in df.columns:
    df = pd.merge(df, seasonality, on=['Distributor_ID', 'Year', 'Month'], how='left')
    df['Seasonality_Score'] = df['Seasonality_Score'].fillna(1)

print("Applying Preprocessing Rules...")

# Identify spatial columns
spatial_cols = [c for c in df.columns if 'dist_nearest' in c or 'density' in c]

if spatial_cols:
    # Rule 1: Spatial Nulls
    df[spatial_cols] = df[spatial_cols].fillna(0)

    # Rule 2: Multicollinearity Check
    corr_matrix = df[spatial_cols].corr(method='pearson').abs()
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    redundant_cols = [c for c in upper_triangle.columns if any(upper_triangle[c] > 0.95)]

    if redundant_cols:
        print(f"Dropping redundant spatial features: {redundant_cols}")
        df.drop(columns=redundant_cols, inplace=True)

# Rule 3: Categorical Casting
text_cols = df.select_dtypes(include=['object', 'string']).columns
for col in text_cols:
    df[col] = df[col].astype('category')

# Save to Gold Layer inside repo data folder
output_path = BASE_DIR / 'gold' / 'master_training_matrix.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print(f"SUCCESS: Feature matrix generated at {output_path}")

Loading Data Layers...
Merging Feature Matrix...
Applying Preprocessing Rules...
Dropping redundant spatial features: ['dist_nearest_schools', 'dist_nearest_hospitals', 'dist_nearest_worship', 'dist_nearest_financial', 'dist_nearest_industrial', 'dist_nearest_markets', 'dist_nearest_tourism', 'dist_nearest_offices']
SUCCESS: Feature matrix generated at /content/drive/MyDrive/data_storm/data/gold/master_training_matrix.csv
